# ChargeSquare Analytics — A1–A6

`energy_kwh` is the **cumulative** per-session meter register — the classic **energy trap**.
Every energy metric below uses per-session deltas, **never** `SUM(energy_kwh)`, which would
over-count by ~(readings per session).

This notebook runs the six analytical queries in `analytics/queries/` against ClickHouse
(over its HTTP interface, port 8123) and writes each result to `analytics/output/*.csv`.


In [1]:
import os
from pathlib import Path
import clickhouse_connect
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Resolve the pipeline root robustly: honour $PIPELINE_ROOT if set, otherwise walk
# upward from the current directory until we find the one that holds BOTH
# docker-compose.yml and analytics/queries. Works from the repo root, from analytics/,
# or from any nested subdirectory.
def find_root():
    env = os.environ.get("PIPELINE_ROOT")
    if env:
        return Path(env).resolve()
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "docker-compose.yml").exists() and (d / "analytics/queries").is_dir():
            return d
    raise RuntimeError(
        "could not locate pipeline root (need docker-compose.yml + analytics/queries); "
        "set PIPELINE_ROOT to override"
    )

ROOT = find_root()
QUERIES = ROOT / "analytics/queries"
OUTPUT = ROOT / "analytics/output"
OUTPUT.mkdir(parents=True, exist_ok=True)

client = clickhouse_connect.get_client(
    host=os.environ.get("CLICKHOUSE_HOST", "localhost"),
    port=int(os.environ.get("CLICKHOUSE_PORT", "8123")),
    username="chargesquare", password="chargesquare", database="ev",
)

LABELS = ["A1", "A2", "A3", "A4", "A5", "A6"]

def run(label):
    # clickhouse-connect wants a single statement, so strip the trailing ';'.
    sql = sorted(QUERIES.glob(f"{label}_*.sql"))[0].read_text(encoding="utf-8").strip().rstrip(";")
    df = client.query_df(sql)
    df.to_csv(OUTPUT / f"{label}.csv", index=False)
    print(f"{label}: {len(df)} rows -> analytics/output/{label}.csv")
    return df

def save_plot(name):
    try:
        plt.tight_layout(); plt.savefig(OUTPUT / f"{name}.png"); plt.show()
    except Exception as e:
        print("plot skipped:", e)
    finally:
        plt.close("all")


## A1 — Hourly energy consumption (per-session deltas, not cumulative sums)


In [2]:
a1 = run("A1")
try:
    a1.plot(x="hour", y="energy_kwh", figsize=(10,3), legend=False, title="A1 Hourly energy (kWh)")
    save_plot("A1")
except Exception as e:
    print("plot skipped:", e)
a1.head()


A1: 14 rows -> analytics/output/A1.csv


/var/folders/88/5j93ks7j4v3d_pf8hqr0_prr0000gn/T/ipykernel_82241/741225504.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT / f"{name}.png"); plt.show()


,hour,energy_kwh
0,2026-07-02 21:00:00,5471.164
1,2026-07-02 22:00:00,27268.220
2,2026-07-02 23:00:00,26488.918
3,2026-07-03 00:00:00,26715.208
4,2026-07-03 01:00:00,26558.199


## A2 — Station uptime / downtime, worst stations per operator


In [3]:
a2 = run("A2")
try:
    top = a2.head(12).iloc[::-1]
    top.plot(kind="barh", x="station_id", y="downtime_s", figsize=(10,4), legend=False, title="A2 Downtime seconds (top stations)")
    save_plot("A2")
except Exception as e:
    print("plot skipped:", e)
a2.head(12)


A2: 20 rows -> analytics/output/A2.csv


/var/folders/88/5j93ks7j4v3d_pf8hqr0_prr0000gn/T/ipykernel_82241/741225504.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT / f"{name}.png"); plt.show()


,operator_id,station_id,total_s,downtime_s,uptime_ratio
0,Esarj,TR-IZM-0017,184496,9037,0.9510
1,ChargeSquare,TR-IZM-0003,184496,6932,0.9624
2,VoltRun,TR-IST-0033,138372,6752,0.9512
3,VoltRun,TR-ADA-0004,184496,6091,0.9670
4,VoltRun,TR-IST-0074,138372,5845,0.9578
5,ZES,TR-IZM-0030,138372,5463,0.9605
6,ZES,TR-ANT-0015,184496,5170,0.9720
7,VoltRun,TR-IZM-0027,184496,5022,0.9728
8,Trugo,TR-IST-0035,184496,4686,0.9746
9,ChargeSquare,TR-ANK-0005,184496,4647,0.9748


## A3 — Average duration & energy by vehicle brand


In [4]:
a3 = run("A3")
try:
    a3.plot(kind="bar", x="brand", y=["avg_duration_min","avg_energy_kwh"], figsize=(10,3), title="A3 Avg duration (min) & energy (kWh) by brand")
    save_plot("A3")
except Exception as e:
    print("plot skipped:", e)
a3


A3: 9 rows -> analytics/output/A3.csv


/var/folders/88/5j93ks7j4v3d_pf8hqr0_prr0000gn/T/ipykernel_82241/741225504.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT / f"{name}.png"); plt.show()


,brand,sessions,avg_duration_min,avg_energy_kwh
0,Tesla,4106,27.4,33.26
1,Volvo,1574,30.9,34.13
2,Volkswagen,969,31.8,34.94
3,Hyundai,765,28.6,36.60
4,Togg,573,32.8,40.17
5,Kia,555,29.4,36.89
6,BMW,536,30.3,37.23
7,Renault,466,30.1,29.64
8,Mercedes-Benz,419,34.6,30.66


## A4 — Revenue and peak-rate contribution, per operator


In [5]:
a4 = run("A4")
try:
    a4.plot(kind="bar", x="operator_id", y=["revenue_eur","peak_revenue_eur"], figsize=(10,3), title="A4 Revenue (EUR): total vs peak-rate")
    save_plot("A4")
except Exception as e:
    print("plot skipped:", e)
a4


A4: 132 rows -> analytics/output/A4.csv


/var/folders/88/5j93ks7j4v3d_pf8hqr0_prr0000gn/T/ipykernel_82241/741225504.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT / f"{name}.png"); plt.show()


,operator_id,city,tariff_id,revenue_eur,peak_revenue_eur,peak_pct,sessions
0,ChargeSquare,Istanbul,standard-v1,6215.29,0.00,0.0,438
1,VoltRun,Istanbul,standard-v1,5645.87,0.00,0.0,393
2,Esarj,Istanbul,standard-v1,4821.86,0.00,0.0,343
3,ZES,Istanbul,standard-v1,4674.35,0.00,0.0,384
4,Trugo,Istanbul,standard-v1,4035.01,0.00,0.0,314
...,...,...,...,...,...,...,...
127,Trugo,Antalya,fleet-v1,82.48,0.00,0.0,5
128,VoltRun,Bursa,peak-rate-v2,81.55,81.55,100.0,4
129,ZES,Adana,peak-rate-v2,79.86,50.37,63.1,3
130,ZES,Adana,fleet-v1,64.89,0.00,0.0,4


## A5 — Geographic distribution of FAULT events (deduped by event_id)


In [6]:
a5 = run("A5")
try:
    a5.plot(kind="bar", x="city", y="fault_count", figsize=(10,3), legend=False, title="A5 Fault count by city")
    save_plot("A5")
except Exception as e:
    print("plot skipped:", e)
a5


A5: 7 rows -> analytics/output/A5.csv


/var/folders/88/5j93ks7j4v3d_pf8hqr0_prr0000gn/T/ipykernel_82241/741225504.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT / f"{name}.png"); plt.show()


,city,fault_count,stations_affected,faults_per_station,lat,lon
0,Istanbul,128,68,1.88,41.0007,28.9796
1,Izmir,69,26,2.65,38.4158,27.1381
2,Ankara,48,26,1.85,39.9142,32.8645
3,Antalya,26,12,2.17,36.8869,30.6954
4,Adana,20,7,2.86,37.0240,35.3418
5,Bursa,12,6,2.00,40.1700,29.0383
6,Konya,9,5,1.80,37.8641,32.5098


## A6 — Anomaly detection: sessions > 2 sigma above the fleet mean power


In [7]:
a6 = run("A6")
try:
    a6.head(20).plot(kind="bar", x="session_id", y="z_score", figsize=(10,3), legend=False, title="A6 Anomalous sessions (z-score)")
    save_plot("A6")
except Exception as e:
    print("plot skipped:", e)
a6.head(20)


A6: 100 rows -> analytics/output/A6.csv


/var/folders/88/5j93ks7j4v3d_pf8hqr0_prr0000gn/T/ipykernel_82241/741225504.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT / f"{name}.png"); plt.show()


,session_id,station_id,brand,avg_power_kw,z_score
0,sess-71b629bf-c818-4559,TR-ANT-0015,Tesla,250.0,3.09
1,sess-6ed9d286-a79f-45b7,TR-IST-0083,Tesla,250.0,3.09
2,sess-aa01e2b5-2a8a-418a,TR-ANT-0012,Tesla,250.0,3.09
3,sess-d9b7faab-bf20-496f,TR-IST-0025,Tesla,250.0,3.09
4,sess-6a59929b-a80e-4074,TR-IZM-0017,Tesla,250.0,3.09
5,sess-24a18607-8a7d-4081,TR-IST-0001,Tesla,250.0,3.09
6,sess-07acbe50-731b-4608,TR-IZM-0019,Tesla,250.0,3.09
7,sess-14bd23cf-2438-4349,TR-IZM-0015,Tesla,250.0,3.09
8,sess-72bcb51f-5680-423f,TR-ANK-0032,Tesla,250.0,3.09
9,sess-f0e265a0-a594-4311,TR-IZM-0014,Tesla,250.0,3.09


## Summary

All six results are in `analytics/output/*.csv`. The energy figures (A1, A3) use
per-session **deltas** of the cumulative `energy_kwh` register — summing the raw column
would over-count by roughly the number of meter readings per session.
